## **[Instituto Mexicano de Tecnología del Agua](https://www.gob.mx/imta).<br>**

### M. en C. Omar Ulises Robles Pereyra

<img src="./Datos/Imagenes/Logos.png" height="100" align="middle">

# Graficación de residuales
Esta libreta grafica los residuales para cada campo de una simulación LES-WALE.
### **Bibliografía**
[1]

---

In [ ]:
#%pip install matplotlib
#%pip install numpy
#%pip install pandas

In [ ]:
from pathlib import Path
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Ruta al archivo de log de OpenFOAM a procesar
RUTA_LOG = Path("Datos/Maiz/log.foamRun")

In [ ]:
# Expresión regular para detectar el tiempo en el log de OpenFOAM
PATRON_TIEMPO = re.compile(
    r"^Time\s+=\s+([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)s?\s*$"
)
# Expresión regular para detectar los residuales en el log de OpenFOAM
PATRON_RESIDUAL = re.compile(
    r"^(?P<solver>[^:]+):\s+Solving for (?P<campo>[^,]+),\s+"
    r"Initial residual = (?P<inicial>[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?),\s+"
    r"Final residual = (?P<final>[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?),\s+"
    r"No Iterations (?P<iteraciones>\d+)\s*$"
)

In [ ]:
# Función para leer residuales de un log de OpenFOAM
def leer_residuales(ruta: Path) -> pd.DataFrame:
    """Extrae los residuales de un log de OpenFOAM sin cargarlo completo en memoria."""
    tiempo = None
    registros = []

    with ruta.open(encoding="utf-8", errors="replace") as archivo:
        for linea in archivo:
            coincidencia_tiempo = PATRON_TIEMPO.match(linea)
            if coincidencia_tiempo:
                tiempo = float(coincidencia_tiempo.group(1))
                continue

            coincidencia_residual = PATRON_RESIDUAL.match(linea)
            if coincidencia_residual and tiempo is not None:
                datos = coincidencia_residual.groupdict()
                registros.append(
                    {
                        "tiempo": tiempo,
                        "solver": datos["solver"],
                        "campo": datos["campo"],
                        "residual_inicial": float(datos["inicial"]),
                        "residual_final": float(datos["final"]),
                        "iteraciones": int(datos["iteraciones"]),
                    }
                )

    if not registros:
        raise ValueError(f"No se encontraron residuales en: {ruta}")

    return pd.DataFrame(registros)

In [ ]:
residuales = leer_residuales(RUTA_LOG)

print(f"Soluciones extraídas: {len(residuales):,}")
print(f"Intervalo de tiempo: {residuales.tiempo.min():g} a {residuales.tiempo.max():g} s")
print("\nSoluciones por campo:")
print(residuales.groupby("campo").size())

In [ ]:
print(residuales)

In [ ]:
""" # Grafica todos los residuales encontrados en el log.
campos = sorted(residuales["campo"].unique())
columnas = 2
filas = int(np.ceil(len(campos) / columnas))

figura, ejes = plt.subplots(
    filas,
    columnas,
    figsize=(14, 4 * filas),
    sharex=True,
)

for eje, campo in zip(np.ravel(ejes), campos):
    datos_campo = residuales.loc[residuales["campo"] == campo]

    eje.semilogy(
        datos_campo["tiempo"],
        datos_campo["residual_inicial"],
        ".",
        alpha=0.45,
        label="Inicial",
    )
    eje.semilogy(
        datos_campo["tiempo"],
        datos_campo["residual_final"],
        ".",
        alpha=0.45,
        label="Final",
    )

    eje.set_title(campo)
    eje.set_xlabel("Tiempo físico [s]")
    eje.set_ylabel("Residual")
    eje.grid(True, which="both", alpha=0.3)
    eje.legend()

for eje in np.ravel(ejes)[len(campos):]:
    eje.remove()

figura.suptitle("Evolución de residuales por solución", y=1.02)
figura.tight_layout()
plt.show()


# Resume las correcciones internas de cada campo por paso de tiempo.
resumen = (
    residuales.groupby(["tiempo", "campo"], as_index=False)
    .agg(
        residual_inicial=("residual_inicial", "max"),
        residual_final=("residual_final", "max"),
        soluciones=("campo", "size"),
    )
)

figura, ejes = plt.subplots(1, 2, figsize=(15, 5), sharex=True, sharey=True)

for campo, datos_campo in resumen.groupby("campo"):
    ejes[0].semilogy(
        datos_campo["tiempo"],
        datos_campo["residual_inicial"],
        label=campo,
    )
    ejes[1].semilogy(
        datos_campo["tiempo"],
        datos_campo["residual_final"],
        label=campo,
    )

for eje, titulo in zip(
    ejes,
    ["Residual inicial máximo", "Residual final máximo"],
):
    eje.set_title(titulo)
    eje.set_xlabel("Tiempo físico [s]")
    eje.set_ylabel("Residual")
    eje.grid(True, which="both", alpha=0.3)
    eje.legend(title="Campo")

figura.suptitle("Evolución resumida de residuales", y=1.02)
figura.tight_layout()
plt.show() """